In [5]:
import pandas as pd
import numpy as np
import shap

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    average_precision_score
)

# 1. 데이터 불러오기
df = pd.read_csv("ai4i2020.csv")

# 2. 입력/정답 설정
X = df[
    [
        "Type",
        "Air temperature [K]",
        "Process temperature [K]",
        "Rotational speed [rpm]",
        "Torque [Nm]",
        "Tool wear [min]"
    ]
].copy()

X = pd.get_dummies(X, columns=["Type"], drop_first=False)

y = df["Machine failure"]

# 3. train / test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# 4. Random Forest
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)

rf.fit(X_train, y_train)

# 5. 성능 확인
pred = rf.predict(X_test)
prob = rf.predict_proba(X_test)[:, 1]

print("Confusion Matrix")
print(confusion_matrix(y_test, pred))

print("\nClassification Report")
print(classification_report(y_test, pred, digits=3))

print(
    "\nPR-AUC:",
    average_precision_score(y_test, prob)
)

# 6. SHAP 계산
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

print("\nSHAP 계산 완료")

Confusion Matrix
[[1916   16]
 [  24   44]]

Classification Report
              precision    recall  f1-score   support

           0      0.988     0.992     0.990      1932
           1      0.733     0.647     0.688        68

    accuracy                          0.980      2000
   macro avg      0.860     0.819     0.839      2000
weighted avg      0.979     0.980     0.979      2000


PR-AUC: 0.7598300753982006

SHAP 계산 완료


In [6]:
print(type(shap_values))
print(np.array(shap_values).shape)

<class 'numpy.ndarray'>
(2000, 8, 2)


In [7]:
sv_fault = shap_values[:, :, 1]

importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance": np.abs(sv_fault).mean(axis=0)
}).sort_values(
    "importance",
    ascending=False
)

display(importance)

,feature,importance
3,Torque [Nm],0.160692
2,Rotational speed [rpm],0.141584
4,Tool wear [min],0.105992
0,Air temperature [K],0.072553
1,Process temperature [K],0.034054
6,Type_L,0.012688
7,Type_M,0.005300
5,Type_H,0.002019


In [8]:
# 고장 확률이 가장 높은 테스트 샘플 선택
idx = prob.argmax()

sample = X_test.iloc[idx]
sample_prob = prob[idx]

# 해당 샘플의 SHAP 값
sample_shap = sv_fault[idx]

sample_importance = pd.DataFrame({
    "feature": X_test.columns,
    "shap_value": sample_shap
}).sort_values(
    "shap_value",
    key=lambda x: abs(x),
    ascending=False
)

top_features = sample_importance.head(5)

agent_input = {
    "machine_failure_probability": round(float(sample_prob), 4),
    "risk_level": "HIGH" if sample_prob >= 0.7 else "MEDIUM",
    "top_features": [
        {
            "feature": row["feature"],
            "value": float(sample[row["feature"]]),
            "shap_value": round(float(row["shap_value"]), 4)
        }
        for _, row in top_features.iterrows()
    ]
}

agent_input

{'machine_failure_probability': 0.94,
 'risk_level': 'HIGH',
 'top_features': [{'feature': 'Rotational speed [rpm]',
   'value': 2737.0,
   'shap_value': 0.2701},
  {'feature': 'Torque [Nm]', 'value': 8.8, 'shap_value': 0.2691},
  {'feature': 'Air temperature [K]', 'value': 298.9, 'shap_value': -0.0548},
  {'feature': 'Tool wear [min]', 'value': 142.0, 'shap_value': -0.0448},
  {'feature': 'Process temperature [K]',
   'value': 310.2,
   'shap_value': -0.0151}]}

In [9]:
# 고장확률이 가장 높았던 테스트 샘플의 원본 행 인덱스
original_idx = X_test.index[idx]

# 실제 고장 여부 + 고장 유형 확인
actual_failure = df.loc[
    original_idx,
    ["Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"]
]

print("원본 인덱스:", original_idx)
print(actual_failure)

원본 인덱스: 1391
Machine failure    1
TWF                0
HDF                0
PWF                1
OSF                0
RNF                0
Name: 1391, dtype: int64


In [10]:
agent_input = {
    "sample_id": int(original_idx),
    "machine_failure_probability": round(float(sample_prob), 4),
    "risk_level": "HIGH",

    "top_features": [
        {
            "feature": row["feature"],
            "value": float(sample[row["feature"]]),
            "shap_value": round(float(row["shap_value"]), 4)
        }
        for _, row in top_features.iterrows()
    ],

    "ground_truth": {
        "machine_failure": int(df.loc[original_idx, "Machine failure"]),
        "failure_type": "PWF"
    }
}

agent_input

{'sample_id': 1391,
 'machine_failure_probability': 0.94,
 'risk_level': 'HIGH',
 'top_features': [{'feature': 'Rotational speed [rpm]',
   'value': 2737.0,
   'shap_value': 0.2701},
  {'feature': 'Torque [Nm]', 'value': 8.8, 'shap_value': 0.2691},
  {'feature': 'Air temperature [K]', 'value': 298.9, 'shap_value': -0.0548},
  {'feature': 'Tool wear [min]', 'value': 142.0, 'shap_value': -0.0448},
  {'feature': 'Process temperature [K]',
   'value': 310.2,
   'shap_value': -0.0151}],
 'ground_truth': {'machine_failure': 1, 'failure_type': 'PWF'}}

In [11]:
{
  "sample_id": 1391,
  "machine_failure_probability": 0.94,
  "risk_level": "HIGH",
  "top_features": [
    {
      "feature": "Rotational speed [rpm]",
      "value": 2737.0,
      "shap_value": 0.2701
    },
    {
      "feature": "Torque [Nm]",
      "value": 8.8,
      "shap_value": 0.2691
    }
  ],
  "ground_truth": {
    "machine_failure": 1,
    "failure_type": "PWF"
  }
}


{'sample_id': 1391,
 'machine_failure_probability': 0.94,
 'risk_level': 'HIGH',
 'top_features': [{'feature': 'Rotational speed [rpm]',
   'value': 2737.0,
   'shap_value': 0.2701},
  {'feature': 'Torque [Nm]', 'value': 8.8, 'shap_value': 0.2691}],
 'ground_truth': {'machine_failure': 1, 'failure_type': 'PWF'}}

In [12]:
{
  "sample_id": 1391,
  "machine_failure_probability": 0.94,
  "risk_level": "HIGH",
  "top_features": [...]
}

{'sample_id': 1391,
 'machine_failure_probability': 0.94,
 'risk_level': 'HIGH',
 'top_features': [Ellipsis]}

In [13]:
agent_input = {
    "sample_id": int(original_idx),
    "machine_failure_probability": round(float(sample_prob), 4),
    "risk_level": "HIGH",

    "top_features": [
        {
            "feature": row["feature"],
            "value": float(sample[row["feature"]]),
            "shap_value": round(float(row["shap_value"]), 4)
        }
        for _, row in top_features.iterrows()
    ]
}

agent_input


{'sample_id': 1391,
 'machine_failure_probability': 0.94,
 'risk_level': 'HIGH',
 'top_features': [{'feature': 'Rotational speed [rpm]',
   'value': 2737.0,
   'shap_value': 0.2701},
  {'feature': 'Torque [Nm]', 'value': 8.8, 'shap_value': 0.2691},
  {'feature': 'Air temperature [K]', 'value': 298.9, 'shap_value': -0.0548},
  {'feature': 'Tool wear [min]', 'value': 142.0, 'shap_value': -0.0448},
  {'feature': 'Process temperature [K]',
   'value': 310.2,
   'shap_value': -0.0151}]}

In [14]:
import json

with open("agent_input_sample_1391.json", "w", encoding="utf-8") as f:
    json.dump(
        agent_input,
        f,
        ensure_ascii=False,
        indent=2
    )

print("저장 완료")

저장 완료
